# Final Maternal Health Preprocessing & Integration Pipeline

This notebook starts from two already prepared input files:

1. `maternal_health_cleaned2.csv`  
   The final cleaned real UCI dataset after:
   - exact duplicate removal,
   - removal of the abnormal heart-rate observation,
   - removal of the 71 rows belonging to conflicting feature combinations.

2. `saudi_maternal_health_risks.csv`  
   The generated Saudi-context synthetic dataset.

The notebook **does not redo the original UCI cleaning**. Instead, it validates both final inputs, cleans/validates the Saudi synthetic data if necessary, creates shared engineered features, and produces a combined modeling dataset using only variables genuinely available in both datasets.

### Final outputs

Running the notebook from top to bottom creates:

- `processed/original_maternal_cleaned2_validated.csv`
- `processed/saudi_synthetic_cleaned_full.csv`
- `processed/maternal_modeling_common_features.csv`

### Important methodology decision

`Pre_pregnancy_BMI` and `Parity` are retained in the Saudi synthetic dataset but are **not imputed into the real UCI patients**, because those measurements were never present in the original dataset.

For final model evaluation, a portion of the **real data should remain an untouched test set**, while synthetic observations may be used to augment training.


## 1. Imports and file locations


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

# The notebook searches common project locations automatically.
SEARCH_DIRS = [
    Path.cwd(),
    Path.cwd() / "dataset",
    Path.cwd().parent / "dataset",
    Path.cwd() / "ml" / "dataset",
]

ORIGINAL_CANDIDATES = [
    "maternal_health_cleaned2.csv",
]

SAUDI_CANDIDATES = [
    "saudi_maternal_health_risks.csv",
]

def find_file(candidates):
    for folder in SEARCH_DIRS:
        for name in candidates:
            path = folder / name
            if path.exists():
                return path
    searched = [str(folder / name) for folder in SEARCH_DIRS for name in candidates]
    raise FileNotFoundError(
        "Could not find the required CSV.\nSearched:\n" + "\n".join(searched)
    )

original_path = find_file(ORIGINAL_CANDIDATES)
saudi_path = find_file(SAUDI_CANDIDATES)

print("Original cleaned dataset:", original_path)
print("Saudi synthetic dataset:", saudi_path)


## 2. Load and standardize both datasets


In [ ]:
original_clean = pd.read_csv(original_path)
saudi_raw = pd.read_csv(saudi_path)

original_clean.columns = original_clean.columns.str.strip()
saudi_raw.columns = saudi_raw.columns.str.strip()

required_original = [
    "Age", "SystolicBP", "DiastolicBP",
    "BS", "BodyTemp", "HeartRate", "RiskLevel"
]

required_saudi = [
    "Age", "Pre_pregnancy_BMI", "Parity",
    "SystolicBP", "DiastolicBP",
    "BS", "BodyTemp", "HeartRate", "RiskLevel"
]

missing_original_cols = [c for c in required_original if c not in original_clean.columns]
missing_saudi_cols = [c for c in required_saudi if c not in saudi_raw.columns]

if missing_original_cols:
    raise ValueError(f"Original cleaned dataset is missing columns: {missing_original_cols}")

if missing_saudi_cols:
    raise ValueError(f"Saudi dataset is missing columns: {missing_saudi_cols}")

# Normalize labels.
original_clean["RiskLevel"] = (
    original_clean["RiskLevel"].astype(str).str.strip().str.lower()
)

saudi_raw["RiskLevel"] = (
    saudi_raw["RiskLevel"].astype(str).str.strip().str.lower()
)

allowed_risks = {"low risk", "mid risk", "high risk"}

# Add provenance to real data.
original_clean["data_source"] = "original"

print("Original cleaned shape:", original_clean.shape)
print("Saudi raw shape:", saudi_raw.shape)

print("\nOriginal missing values:", int(original_clean.isna().sum().sum()))
print("Saudi missing values:", int(saudi_raw.isna().sum().sum()))

print("\nOriginal RiskLevel values:", sorted(original_clean["RiskLevel"].unique()))
print("Saudi RiskLevel values:", sorted(saudi_raw["RiskLevel"].unique()))


## 3. Validate the final cleaned original dataset

The file `maternal_health_cleaned2.csv` is already the final real dataset after the agreed preprocessing decisions.

This section therefore **validates rather than repeats** the original cleaning.

Expected checks:
- 380 observations,
- no missing values,
- no exact duplicates,
- no physiologically impossible heart rates,
- no remaining identical feature combinations carrying conflicting `RiskLevel` labels.


In [ ]:
original_feature_cols = [
    "Age", "SystolicBP", "DiastolicBP",
    "BS", "BodyTemp", "HeartRate"
]

# Basic validation
original_duplicate_n = int(original_clean.duplicated(subset=required_original).sum())
original_missing_n = int(original_clean[required_original].isna().sum().sum())

invalid_original_hr = (
    (original_clean["HeartRate"] < 40)
    | (original_clean["HeartRate"] > 180)
).sum()

# Check for remaining conflicting labels
remaining_conflicts_original = (
    original_clean
    .groupby(original_feature_cols)["RiskLevel"]
    .nunique()
)

remaining_conflicts_original = remaining_conflicts_original[
    remaining_conflicts_original > 1
]

print("FINAL ORIGINAL DATASET VALIDATION")
print("-" * 45)
print("Rows:", len(original_clean))
print("Missing values:", original_missing_n)
print("Exact duplicate rows:", original_duplicate_n)
print("Invalid HR rows:", int(invalid_original_hr))
print("Remaining conflicting groups:", len(remaining_conflicts_original))

print("\nClass distribution:")
print(original_clean["RiskLevel"].value_counts())

print("\nClass percentages:")
print(
    (original_clean["RiskLevel"].value_counts(normalize=True) * 100)
    .round(1)
)

assert original_missing_n == 0, "Missing values remain in original cleaned data."
assert original_duplicate_n == 0, "Exact duplicates remain in original cleaned data."
assert invalid_original_hr == 0, "Physiologically invalid HR values remain."
assert len(remaining_conflicts_original) == 0, "Conflicting labels still remain."
assert set(original_clean["RiskLevel"].unique()).issubset(allowed_risks)

print("\n✅ Original cleaned dataset validated successfully.")


### 3.1 Original-data status

If the previous cell completes successfully, `maternal_health_cleaned2.csv` is accepted as the final cleaned real dataset and no additional original-data cleaning is performed in this notebook.


In [ ]:
if len(original_clean) != 380:
    print(
        f"⚠️ Note: expected 380 cleaned real observations based on the team's prior pipeline, "
        f"but found {len(original_clean)}. Continue only if this is intentional."
    )
else:
    print("✅ Expected 380 cleaned real observations confirmed.")


## 4. Validate and clean the synthetic Saudi dataset

The Saudi synthetic dataset is kept separate because it contains additional features (`Pre_pregnancy_BMI` and `Parity`) that were never measured in the original UCI data.

The validation below removes exact duplicates, invalid target/BMI categories, physiologically impossible records, and any conflicting labels.


In [ ]:
saudi_clean = saudi_raw.copy()

# Ensure provenance exists and is consistent.
saudi_clean["data_source"] = "synthetic_saudi"

saudi_start_n = len(saudi_clean)

# Exact duplicates
saudi_duplicate_n = int(saudi_clean.duplicated().sum())
saudi_clean = saudi_clean.drop_duplicates().copy()

# Valid categorical values
allowed_bmi = {"underweight", "normal", "overweight", "obese"}
saudi_clean["Pre_pregnancy_BMI"] = (
    saudi_clean["Pre_pregnancy_BMI"]
    .astype(str)
    .str.strip()
    .str.lower()
)

invalid_category_mask = (
    ~saudi_clean["RiskLevel"].isin(allowed_risks)
    | ~saudi_clean["Pre_pregnancy_BMI"].isin(allowed_bmi)
)

saudi_invalid_category_n = int(invalid_category_mask.sum())
saudi_clean = saudi_clean.loc[~invalid_category_mask].copy()

# Conservative physiological plausibility checks used in the team's notebook.
invalid_saudi_mask = (
    (saudi_clean["Age"] < 15) | (saudi_clean["Age"] > 55)
    | (saudi_clean["Parity"] < 0) | (saudi_clean["Parity"] > 20)
    | (saudi_clean["SystolicBP"] < 70) | (saudi_clean["SystolicBP"] > 200)
    | (saudi_clean["DiastolicBP"] < 40) | (saudi_clean["DiastolicBP"] > 130)
    | (saudi_clean["SystolicBP"] <= saudi_clean["DiastolicBP"])
    | (saudi_clean["BS"] < 2.0) | (saudi_clean["BS"] > 30.0)
    | (saudi_clean["BodyTemp"] < 94) | (saudi_clean["BodyTemp"] > 106)
    | (saudi_clean["HeartRate"] < 40) | (saudi_clean["HeartRate"] > 180)
)

saudi_invalid_physiology_n = int(invalid_saudi_mask.sum())
saudi_clean = saudi_clean.loc[~invalid_saudi_mask].copy()

# Check conflicting labels using ALL Saudi predictor fields.
saudi_feature_cols_full = [
    "Age", "Pre_pregnancy_BMI", "Parity",
    "SystolicBP", "DiastolicBP",
    "BS", "BodyTemp", "HeartRate"
]

saudi_risk_counts = (
    saudi_clean
    .groupby(saudi_feature_cols_full, dropna=False)["RiskLevel"]
    .nunique()
)

saudi_conflicting_profiles = saudi_risk_counts[saudi_risk_counts > 1].index

saudi_conflict_mask = (
    saudi_clean
    .set_index(saudi_feature_cols_full)
    .index
    .isin(saudi_conflicting_profiles)
)

saudi_conflicting_rows_n = int(saudi_conflict_mask.sum())
saudi_conflicting_groups_n = len(saudi_conflicting_profiles)

saudi_clean = saudi_clean.loc[~saudi_conflict_mask].copy()
saudi_clean = saudi_clean.reset_index(drop=True)

print("SAUDI SYNTHETIC DATA CLEANING")
print("-" * 45)
print(f"Rows at start:                    {saudi_start_n}")
print(f"Exact duplicate rows removed:     {saudi_duplicate_n}")
print(f"Invalid category rows removed:    {saudi_invalid_category_n}")
print(f"Physiologically invalid removed:  {saudi_invalid_physiology_n}")
print(f"Conflicting feature groups:       {saudi_conflicting_groups_n}")
print(f"Rows removed due to conflicts:    {saudi_conflicting_rows_n}")
print(f"Final cleaned Saudi rows:         {len(saudi_clean)}")
print(f"Final missing values:             {int(saudi_clean.isna().sum().sum())}")
print(f"Final duplicate rows:             {int(saudi_clean.duplicated().sum())}")

print("\nFinal Saudi class distribution:")
print(saudi_clean["RiskLevel"].value_counts())

print("\nFinal Saudi class percentages:")
print((saudi_clean["RiskLevel"].value_counts(normalize=True) * 100).round(1))


### 4.1 Final Saudi validation assertions


In [ ]:

assert saudi_clean.isna().sum().sum() == 0, "Missing values remain in Saudi data."
assert saudi_clean.duplicated().sum() == 0, "Exact duplicates remain in Saudi data."
assert set(saudi_clean["RiskLevel"].unique()).issubset(allowed_risks)
assert set(saudi_clean["Pre_pregnancy_BMI"].unique()).issubset(allowed_bmi)

remaining_saudi_conflicts = (
    saudi_clean
    .groupby(saudi_feature_cols_full)["RiskLevel"]
    .nunique()
)
remaining_saudi_conflicts = remaining_saudi_conflicts[remaining_saudi_conflicts > 1]

print("Remaining Saudi conflicting groups:", len(remaining_saudi_conflicts))
assert len(remaining_saudi_conflicts) == 0, "Conflicting Saudi labels still remain."


## 5. Create shared engineered features

`Pre_pregnancy_BMI` and `Parity` cannot be added to the real patients because those measurements do not exist in the UCI dataset.

Instead, two transparent features are derived from blood pressure values available in **both** datasets:

- **Pulse Pressure** = SystolicBP − DiastolicBP
- **Mean Arterial Pressure (MAP)** = (SystolicBP + 2 × DiastolicBP) / 3

These are calculated from observed variables rather than invented patient information.


In [ ]:

def add_shared_engineered_features(frame):
    out = frame.copy()
    out["PulsePressure"] = out["SystolicBP"] - out["DiastolicBP"]
    out["MeanArterialPressure"] = (
        out["SystolicBP"] + 2 * out["DiastolicBP"]
    ) / 3
    return out

original_clean = add_shared_engineered_features(original_clean)
saudi_clean = add_shared_engineered_features(saudi_clean)

print(
    original_clean[
        ["SystolicBP", "DiastolicBP", "PulsePressure", "MeanArterialPressure"]
    ].head()
)


## 6. Build the modeling-ready combined dataset

The combined dataset intentionally excludes `Pre_pregnancy_BMI` and `Parity` because those variables were never measured in `maternal_health_cleaned2.csv`.

This avoids creating hundreds of artificial missing values or inventing measurements for real patients.


In [ ]:

shared_predictors = [
    "Age",
    "SystolicBP",
    "DiastolicBP",
    "BS",
    "BodyTemp",
    "HeartRate",
    "PulsePressure",
    "MeanArterialPressure",
]

modeling_cols = shared_predictors + ["RiskLevel", "data_source"]

original_for_modeling = original_clean[modeling_cols].copy()
saudi_for_modeling = saudi_clean[modeling_cols].copy()

combined_modeling = pd.concat(
    [original_for_modeling, saudi_for_modeling],
    ignore_index=True
)

print("Original cleaned rows:", len(original_for_modeling))
print("Saudi synthetic cleaned rows:", len(saudi_for_modeling))
print("Combined modeling rows:", len(combined_modeling))
print("Combined missing values:", int(combined_modeling.isna().sum().sum()))

print("\nRows by source:")
print(combined_modeling["data_source"].value_counts())

print("\nRisk distribution by source:")
print(pd.crosstab(
    combined_modeling["data_source"],
    combined_modeling["RiskLevel"],
    margins=True
))

assert combined_modeling.isna().sum().sum() == 0, "Combined modeling data contains missing values."


### Important modeling note

Do **not** randomly split the entire combined dataset and report that as final performance.

For a defensible evaluation:

1. Split the **real cleaned observations** into real training and real test sets.
2. Keep the real test set untouched.
3. Add synthetic Saudi observations only to the training portion.
4. Evaluate every model on the same untouched real test set.

The `data_source` column is retained specifically to support this design.


## 7. Final preprocessing summaries


In [ ]:
summary = pd.DataFrame({
    "Dataset": [
        "Original cleaned2 (validated)",
        "Saudi synthetic raw",
        "Saudi synthetic cleaned",
        "Combined common-feature dataset"
    ],
    "Rows": [
        len(original_clean),
        len(saudi_raw),
        len(saudi_clean),
        len(combined_modeling)
    ],
    "Missing values": [
        int(original_clean.isna().sum().sum()),
        int(saudi_raw.isna().sum().sum()),
        int(saudi_clean.isna().sum().sum()),
        int(combined_modeling.isna().sum().sum())
    ]
})

summary


In [ ]:

print("FINAL ORIGINAL CLASS DISTRIBUTION")
print(original_clean["RiskLevel"].value_counts())
print((original_clean["RiskLevel"].value_counts(normalize=True) * 100).round(1))

print("\nFINAL SAUDI CLASS DISTRIBUTION")
print(saudi_clean["RiskLevel"].value_counts())
print((saudi_clean["RiskLevel"].value_counts(normalize=True) * 100).round(1))

print("\nCOMBINED CLASS DISTRIBUTION")
print(combined_modeling["RiskLevel"].value_counts())
print((combined_modeling["RiskLevel"].value_counts(normalize=True) * 100).round(1))


## 8. Presentation visualizations


In [ ]:

# Class distribution by source
plot_data = pd.crosstab(
    combined_modeling["data_source"],
    combined_modeling["RiskLevel"]
).reindex(columns=["low risk", "mid risk", "high risk"], fill_value=0)

ax = plot_data.plot(kind="bar", figsize=(9, 5))
ax.set_title("Risk-Level Distribution by Data Source")
ax.set_xlabel("Data source")
ax.set_ylabel("Number of observations")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


In [ ]:

# Combined numeric-feature correlation matrix (matplotlib only)
corr_features = shared_predictors
corr = combined_modeling[corr_features].corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr.values, aspect="auto")

ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index)

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

ax.set_title("Correlation Matrix of Shared Modeling Features")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## 9. Save final datasets

All outputs are written to a `processed` folder beside the notebook's current working directory.


In [ ]:
OUTPUT_DIR = Path.cwd() / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

original_output = OUTPUT_DIR / "original_maternal_cleaned2_validated.csv"
saudi_output = OUTPUT_DIR / "saudi_synthetic_cleaned_full.csv"
combined_output = OUTPUT_DIR / "maternal_modeling_common_features.csv"

original_clean.to_csv(original_output, index=False)
saudi_clean.to_csv(saudi_output, index=False)
combined_modeling.to_csv(combined_output, index=False)

print("Saved:")
print(" -", original_output)
print(" -", saudi_output)
print(" -", combined_output)


## 10. Final verification

If this cell completes without an assertion error, preprocessing is complete.


In [ ]:
# Original
assert original_clean[required_original].isna().sum().sum() == 0
assert original_clean.duplicated(subset=required_original).sum() == 0
assert len(remaining_conflicts_original) == 0

# Saudi
assert saudi_clean.isna().sum().sum() == 0
assert saudi_clean.duplicated().sum() == 0

# Combined common-feature data
assert combined_modeling.isna().sum().sum() == 0
assert set(combined_modeling["data_source"].unique()) == {"original", "synthetic_saudi"}
assert set(combined_modeling["RiskLevel"].unique()).issubset(allowed_risks)

# Output files
assert original_output.exists()
assert saudi_output.exists()
assert combined_output.exists()

print("✅ PREPROCESSING COMPLETE")
print(f"Clean real observations:        {len(original_clean)}")
print(f"Clean synthetic observations:   {len(saudi_clean)}")
print(f"Combined modeling observations: {len(combined_modeling)}")
print("\nMain modeling file:")
print(combined_output)
print("\nFor final evaluation, preserve an untouched REAL-only test set.")
